# CIFAR-10 Hardness-Gated Multi-Seed Runner

Dispatches the CIFAR-10 hardness-gated multi-seed runs:
- **gated_k0_seed123** (FF_SEED=123)
- **gated_k0_seed456** (FF_SEED=456)

Uses the trainer in `trainers/`
`cp_fair_hardness_gated.py`.

### Configuration

The training scripts read `FF_GAMMA_GATING_MODE=adaptive`,
`FF_GAMMA_0=0.7`, `FF_GAMMA_KAPPA=0.0`, `FF_GAMMA_TAU=1.0`, and
`FF_SEED=<seed>` to the gated trainer — no architecture overrides.
The trainer therefore runs at its **reduced-scale defaults** (L=4,
D=128, 180 epochs, 16 experts, top-2, bs=256, SAM on). This notebook
matches the configuration in the README, so these Colab runs are a
reproducible Colab launcher for the multi-seed configurations described in the README.

This config also matches the existing seed-42 gated ablation already
reported in the paper at the L4/D128 scale — so with seeds 123 and 456
added, the `L4, D128` row of `tab:dissociation` (submitted numbering; in the
camera-ready it is `tab:gated_k0_multiseed`, Table 23) is promoted from
single-seed to **n = 3 mean±std** for the gated κ=0 entry.

Per-run cost: ~4–8 h on Colab A100 → **~10–16 h for both runs**.

Checkpoints and metrics are written to Google Drive every 60 s so a
Colab disconnect never loses progress. Re-running the notebook
resumes from whatever is on Drive.

### Prerequisite — upload the patched gated trainer to Drive

Before running this notebook, upload the trainer to:

```
MyDrive/ff_cifar10_gated_multiseed/cp_fair_hardness_gated.py
```

Source:
`supplement_code/trainers/cp_fair_hardness_gated.py`
(already has the `_sum_prev_goodness` locality fix).

### Local / multi-GPU dispatch

Remove the `gated_k0_seed123` and `gated_k0_seed456` lines from the
`JOBS=( ... )` array in `run_cifar10_multiseed.sh` so the local
dispatcher does not double-run them after `gamma0_seed456` finishes.
(The currently running bash has already read `JOBS` into memory, so
also `kill` any new gated child process that appears when
`gamma0_seed456` completes.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, subprocess, shutil, time, threading, sys
from datetime import datetime

DRIVE_BASE = '/content/drive/MyDrive/ff_cifar10_gated_multiseed'
WORK_DIR = '/content/ff_cifar10_gated'
SCRIPT_NAME = 'cp_fair_hardness_gated.py'
SCRIPT_DRIVE = f'{DRIVE_BASE}/{SCRIPT_NAME}'
SCRIPT_LOCAL = f'{WORK_DIR}/{SCRIPT_NAME}'
PROGRESS_FILE = f'{DRIVE_BASE}/training_progress.json'

os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.exists(SCRIPT_DRIVE), (
    f'Upload the patched gated trainer to {SCRIPT_DRIVE} before running.'
)
shutil.copy2(SCRIPT_DRIVE, SCRIPT_LOCAL)
print(f'Drive base:  {DRIVE_BASE}')
print(f'Work dir:    {WORK_DIR}')
print(f'Script:      {SCRIPT_LOCAL}')


## 1. Run Configs

Two runs only: `gated_k0_seed123` and `gated_k0_seed456`. No
architecture overrides are set, so the trainer falls through to its
reduced-scale defaults (L=4, D=128, 180 epochs, 16 experts, top-2,
bs=256, SAM on) — matching the configuration in the README.

In [ ]:
# ===================== CHANGE HERE IF NEEDED =====================
# COMMON_ENV is intentionally empty of architecture overrides. The
# hardness-gated trainer's defaults (D=128, 180 ep, 16 experts, top-2,
# bs=256, SAM on) match the configuration in the README and the seed-42
# gated ablation. FF_USE_SAM=1 is set explicitly for clarity.
COMMON_ENV = {
    'FF_USE_SAM': '1',
}

# Two hardness-gated κ=0 runs at seeds 123 and 456.
RUN_CONFIGS = [
    ('gated_k0_seed123', {
        'FF_GAMMA_GATING_MODE': 'adaptive',
        'FF_GAMMA_0':    '0.7',
        'FF_GAMMA_KAPPA':'0.0',
        'FF_GAMMA_TAU':  '1.0',
        'FF_SEED':       '123',
    }),
    ('gated_k0_seed456', {
        'FF_GAMMA_GATING_MODE': 'adaptive',
        'FF_GAMMA_0':    '0.7',
        'FF_GAMMA_KAPPA':'0.0',
        'FF_GAMMA_TAU':  '1.0',
        'FF_SEED':       '456',
    }),
]

print(f'{len(RUN_CONFIGS)} runs queued (reduced-scale defaults):')
for tag, env in RUN_CONFIGS:
    print(f'  - {tag:22s}  seed={env["FF_SEED"]:<4}  gated kappa={env["FF_GAMMA_KAPPA"]}')


## 2. Helpers (progress, accuracy parsing, done-detection)

In [ ]:
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {}

def save_progress(p):
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(p, f, indent=2)

def extract_accuracy(log_path):
    """Parse final test accuracy (Top1, Top5) for Stage-1/Stage-2 from the log."""
    r = {}
    if not os.path.exists(log_path):
        return r
    with open(log_path) as f:
        lines = f.readlines()
    for line in reversed(lines):
        if 'Stage-2 Head' in line and 'Top1=' in line:
            try:
                top1 = line.split('Top1=')[1].split('%')[0] + '%'
                top5 = line.split('Top5=')[1].split('%')[0] + '%'
                if 'no-TTA' in line:
                    r['s2_top1_notta'] = top1; r['s2_top5_notta'] = top5
                elif 'TTA' in line:
                    r['s2_top1_tta'] = top1; r['s2_top5_tta'] = top5
            except Exception:
                pass
        if 'Stage-1 FF' in line and 'TTA' in line and 'no-TTA' not in line and 'Top1=' in line:
            try:
                r['s1_top1_tta'] = line.split('Top1=')[1].split('%')[0] + '%'
                r['s1_top5_tta'] = line.split('Top5=')[1].split('%')[0] + '%'
            except Exception:
                pass
        if len(r) >= 6:
            break
    return r

def is_done(tag):
    ckpt_dir = f'{DRIVE_BASE}/checkpoints_{tag}'
    if os.path.exists(ckpt_dir):
        if any('stage2_best' in f for f in os.listdir(ckpt_dir)):
            return True
    log = f'{DRIVE_BASE}/{tag}_stdout.log'
    return 's2_top1_tta' in extract_accuracy(log) if os.path.exists(log) else False

# Status of each run
progress = load_progress()
print('Status of the 2 runs:')
for tag, _ in RUN_CONFIGS:
    status = 'DONE' if is_done(tag) else progress.get(tag, {}).get('status', 'pending')
    acc = progress.get(tag, {}).get('s2_top1_tta', '---')
    print(f'  {tag:22s} [{status:<10s}] S2-Top1-TTA: {acc}')


## 3. DriveSyncer + Training Loop

`DriveSyncer` is a background thread that copies local JSONL/JSON/NPZ
metrics to Drive every 60 s. Checkpoints are written directly to Drive
by the trainer (via `FF_CKPT_DIR`). This survives a Colab disconnect:
on re-run, `is_done(tag)` sees the `stage2_best.pt` on Drive and skips
the run.

In [ ]:
class DriveSyncer(threading.Thread):
    def __init__(self, local_dir, drive_dir, log_local, log_drive, interval=60):
        super().__init__(daemon=True)
        self.local_dir = local_dir
        self.drive_dir = drive_dir
        self.log_local = log_local
        self.log_drive = log_drive
        self.interval = interval
        self._stop = threading.Event()

    def _sync(self):
        try:
            for root, dirs, files in os.walk(self.local_dir):
                rel = os.path.relpath(root, self.local_dir)
                dst_root = os.path.join(self.drive_dir, rel)
                os.makedirs(dst_root, exist_ok=True)
                for fn in files:
                    if fn.endswith(('.jsonl', '.json', '.npz')):
                        shutil.copy2(os.path.join(root, fn), os.path.join(dst_root, fn))
            if os.path.exists(self.log_local):
                shutil.copy2(self.log_local, self.log_drive)
        except Exception as e:
            print(f'[DriveSyncer] sync error (will retry): {e}')

    def run(self):
        while not self._stop.is_set():
            self._stop.wait(self.interval)
            if not self._stop.is_set():
                self._sync()

    def stop(self):
        self._stop.set()
        self._sync()


def run_one(tag, env_overrides):
    ckpt_dir  = f'{DRIVE_BASE}/checkpoints_{tag}'
    runs_dir  = f'{WORK_DIR}/runs_{tag}'                # LOCAL (fast writes)
    runs_drv  = f'{DRIVE_BASE}/runs_{tag}'              # DRIVE (sync target)
    log_drv   = f'{DRIVE_BASE}/{tag}_stdout.log'
    log_loc   = f'{WORK_DIR}/{tag}_stdout.log'
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(runs_dir, exist_ok=True)
    os.makedirs(runs_drv, exist_ok=True)

    env = os.environ.copy()
    env.update(COMMON_ENV)
    env.update(env_overrides)
    env['FF_VERSION_TAG'] = f'cifar10_{tag}'
    env['FF_RUNS_DIR']    = runs_dir
    env['FF_CKPT_DIR']    = ckpt_dir

    existing = [f for f in os.listdir(ckpt_dir) if f.endswith('.pt')]
    print(f'\n{"=" * 64}')
    print(f'  CIFAR-10 Gated Multi-Seed — {tag}')
    seed = env_overrides.get('FF_SEED')
    print(f'  Mode:         gated kappa={env_overrides.get("FF_GAMMA_KAPPA")}, seed={seed}')
    print(f'  Checkpoints:  {ckpt_dir} (Drive)')
    print(f'  Metrics:      {runs_dir} (local) → {runs_drv} every 60s')
    print(f'  Existing ckpts: {existing or "(none, starting fresh)"}')
    print(f'{"=" * 64}\n')

    p = load_progress()
    p[tag] = {
        'status': 'running',
        'started': datetime.now().isoformat(),
        'seed': seed,
        'env': {k: v for k, v in env_overrides.items()},
    }
    save_progress(p)

    syncer = DriveSyncer(runs_dir, runs_drv, log_loc, log_drv, interval=60)
    syncer.start()

    start_time = time.time()
    with open(log_loc, 'a') as lf:
        proc = subprocess.Popen(
            [sys.executable, '-u', SCRIPT_LOCAL], env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
            cwd=WORK_DIR,
        )
        for line in proc.stdout:
            print(line, end='')
            lf.write(line); lf.flush()
        proc.wait()
    elapsed = time.time() - start_time
    syncer.stop()

    acc = extract_accuracy(log_loc)
    p = load_progress()
    p[tag] = {
        'status': 'done' if acc else 'incomplete',
        'seed': seed,
        'finished': datetime.now().isoformat(),
        'elapsed_sec': round(elapsed, 1),
        'return_code': proc.returncode,
        **acc,
    }
    save_progress(p)
    print(f'\n>>> {tag}: {acc or f"rc={proc.returncode}"} (elapsed: {elapsed / 60:.1f} min)')
    return acc


## 4. Run both configs sequentially (skipping done ones)

Re-executing this cell after a disconnect resumes from Drive state: any
run whose `stage2_best.pt` exists on Drive is skipped.

In [ ]:
for tag, env_overrides in RUN_CONFIGS:
    if is_done(tag):
        acc = extract_accuracy(f'{DRIVE_BASE}/{tag}_stdout.log')
        print(f'[skip] {tag}: already complete — {acc}')
        continue
    p = load_progress()
    if p.get(tag, {}).get('status') == 'running':
        print(f'[resume] {tag}: previous run crashed or disconnected — restarting from last checkpoint.')
    run_one(tag, env_overrides)


## 5. Summary table

In [ ]:
import numpy as np

def _num(s, default=None):
    try:
        return float(s.rstrip('%')) if isinstance(s, str) else default
    except Exception:
        return default

progress = load_progress()
rows = []
for tag, _ in RUN_CONFIGS:
    r = progress.get(tag, {})
    rows.append((tag,
                 r.get('status', 'pending'),
                 r.get('s1_top1_tta', '---'),
                 r.get('s2_top1_notta', '---'),
                 r.get('s2_top1_tta', '---')))

print(f'{"run":22s} {"status":<10s} {"S1-TTA":>8s} {"S2-noTTA":>10s} {"S2-TTA":>8s}')
print('-' * 64)
for r in rows:
    print(f'{r[0]:22s} {r[1]:<10s} {r[2]:>8s} {r[3]:>10s} {r[4]:>8s}')

# Compute mean+-std over the gated_k0_seed group (n=2 from Colab;
# combine with seed-42 ablation to reach n=3 in the paper).
print('\n--- group summary (S2 TTA) ---')
vals = []
for tag, _ in RUN_CONFIGS:
    if tag.startswith('gated_k0_seed'):
        v = _num(progress.get(tag, {}).get('s2_top1_tta'))
        if v is not None:
            vals.append(v)
if vals:
    mu = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    print(f'  gated_k0      n={len(vals)}  mean={mu:.2f}%  std={sd:.2f}%')
else:
    print('  gated_k0      (no results yet)')
